<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/%EC%B2%A0%EB%8F%84%EA%B3%84%EC%88%98_%EA%B9%83%ED%97%99_%EC%83%98%ED%94%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# dataset.py
import torch
from dgl.data import DGLDataset
from dgl.dataloading import GraphDataLoader
import numpy as np
import dgl



class Dataset(DGLDataset):
    def __init__(self, type=None, ws=None, split=None):
        super().__init__(name='dataset')
        assert type != None, 'type should be in ["straight", "curve"]'
        assert ws != None, 'Error : window size should NOT be None type!'
        assert split in ['train', 'inference'], 'Dataset split error'

        # init variables
        self.type = type
        self.ws = ws
        self.split = split

        # load data
        self.data = np.load('./data/s_data.npy', allow_pickle=True) if type=='straight' else np.load('./data/c_data.npy', allow_pickle=True)
        if split == 'inference':
            self.data = self.data[10001-ws+1:10001-ws+1+1999]

    def __len__(self):
        if self.split == 'train':
            return 10001-self.ws+1
        else: # 'inference'
            return 1999

    def __getitem__(self, idx):
        distance = torch.zeros((self.ws, 1))
        lane = torch.zeros((self.ws, 5))
        norm_target = torch.zeros((self.ws, 5, 4))
        feature_wheel = torch.zeros((4, self.ws, 5, 8))
        feature_sensor = torch.zeros((2, self.ws , 5, 4))
        graph = dgl.heterograph({('wheel', 'front', 'wheel'): ([0,3],[3,0]),
                                 ('wheel', 'left', 'wheel'): ([0,1],[1,0]),
                                 ('wheel', 'right', 'wheel'): ([3,2],[2,3]),
                                 ('wheel', 'rear', 'wheel'): ([1,2],[2,1]),
                                 ('wheel', 'connect', 'sensor'):([0,1,2,3],[0,0,1,1]),
                                 ('sensor', 'connect', 'wheel'):([0,0,1,1],[0,1,2,3])})
        for i in range(self.ws):
            distance[i] = self.data[idx+i]['distance']
            feature_wheel[:,i,...] = self.data[idx+1]['graph'].nodes['wheel'].data['feature']
            feature_sensor[:,i,...] = self.data[idx+1]['graph'].nodes['sensor'].data['feature']
            lane[i] = self.data[idx+i]['lane'] if self.type=='straight' else torch.zeros(5)
            norm_target[i] = self.data[idx+i]['norm_target']
        graph.nodes['wheel'].data['feature'] = feature_wheel.type(torch.float32)
        graph.nodes['sensor'].data['feature'] = feature_sensor.type(torch.float32)
        distance = distance.type(torch.float32)
        lane = lane.type(torch.float32)
        norm_target = norm_target.type(torch.float32)

        target = self.data[idx+self.ws]['target'].type(torch.float32)
        return distance, lane, graph, norm_target, target



def test_dataset(type, ws, split):
    dataset = Dataset(type, ws, split)
    dataloader = GraphDataLoader(dataset, batch_size=32, shuffle=True, drop_last=False)
    distance, lane, graph, norm_target, target = next(iter(dataloader))
    print(f'======= Test on [type : {type}, ws : {ws}, split : {split}] ======')
    print(f'dataset length when ws is {ws} : {len(dataset)}')
    print(f'distance : {distance.shape}')
    print(f'lane : {lane.shape}')
    print(f'norm_target : {norm_target.shape}')
    print(f'target : {target.shape}')
    print(f'graph/wheel : {graph.nodes["wheel"].data["feature"].shape}')
    print(f'graph/sensor : {graph.nodes["sensor"].data["feature"].shape}')
    print('==================================================================\n')



if __name__ == '__main__':
    test_dataset('straight', 10, 'train')
    test_dataset('straight', 10, 'inference')
    test_dataset('curve', 10, 'train')
    test_dataset('curve', 10, 'inference')
    '''
    ======= Test on [type : straight, ws : 10, split : train] ======
    dataset length when ws is 10 : 9992
    distance : torch.Size([32, 10, 1])
    lane : torch.Size([32, 10, 5])
    norm_target : torch.Size([32, 10, 5, 4])
    target : torch.Size([32, 4])
    graph/wheel : torch.Size([128, 10, 5, 8])
    graph/sensor : torch.Size([64, 10, 5, 4])
    ==================================================================
    '''

ModuleNotFoundError: ignored

In [2]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# infer.py
# library
from tqdm import tqdm
import torch
import pandas as pd
import os
import warnings
import argparse
import json
import numpy as np
import dgl
import copy

# local
from utils import make_dir, AverageMeter
from model import Model
from preprocess import get_mean_std




# target의 mean과 std
s_30 = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv'))
s_40 = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s_50 = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s_70 = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s_100 = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')

c_30 = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c_40 = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c_50 = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c_70 = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c_100 = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

s_30_mean, s_30_std = get_mean_std(s_30) # 모두 shape [4]
s_40_mean, s_40_std = get_mean_std(s_40)
s_50_mean, s_50_std = get_mean_std(s_50)
s_70_mean, s_70_std = get_mean_std(s_70)
s_100_mean, s_100_std = get_mean_std(s_100)

c_30_mean, c_30_std = get_mean_std(c_30) # 모두 shape [4]
c_40_mean, c_40_std = get_mean_std(c_40)
c_50_mean, c_50_std = get_mean_std(c_50)
c_70_mean, c_70_std = get_mean_std(c_70)
c_100_mean, c_100_std = get_mean_std(c_100)

mean_std_dic = {'straight':{30:[s_30_mean, s_30_std],
                            40:[s_40_mean, s_40_std],
                            50:[s_50_mean, s_50_std],
                            70:[s_70_mean, s_70_std],
                            100:[s_100_mean, s_100_std]},
                'curve':{30:[c_30_mean, c_30_std],
                         40:[c_40_mean, c_40_std],
                         50:[c_50_mean, c_50_std],
                         70:[c_70_mean, c_70_std],
                         100:[c_100_mean, c_100_std]}}


mean_std_30 = [AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()]
mean_std_40 = [AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()]
mean_std_50 = [AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()]
mean_std_70 = [AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()]
mean_std_100 = [AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()]
mean_std = [mean_std_30,
            mean_std_40,
            mean_std_50,
            mean_std_70,
            mean_std_100]


# argparse
def get_parser():
    parser = argparse.ArgumentParser(description='KRRI inference')
    parser.add_argument('--experiment_straight', '--es', default='', type=str,
                        help='Straight model experiment name')
    parser.add_argument('--experiment_curve', '--ec', default='', type=str,
                        help='Curve model experiment name')
    parser.add_argument('--avg_type', default='inference', type=str, choices=['train', 'inference'],
                        help='추론 단계에서 사용할 target의 mean과 std가 어느 데이터에서 계산되는가')
    args = parser.parse_args()
    return args



def main(args):
    assert args.experiment_straight != '', '추론할 Straight model의 실험 이름을 지정하세요.'
    assert args.experiment_curve != '', '추론할 Curve model의 실험 이름을 지정하세요.'


    # 경로에 추론결과가 없다면 일단 추론
    if not os.path.exists(f'./exp/{args.experiment_straight}/answer.csv'):
        with open(f'./exp/{args.experiment_straight}/configuration.json', 'r') as f:
            configuration = json.load(f)
        type = configuration['type']
        model = torch.load(f'./exp/{args.experiment_straight}/model.pth').cuda()
        inference(model, configuration['ws'], args.experiment_straight, type)
    if not os.path.exists(f'./exp/{args.experiment_curve}/answer.csv'):
        with open(f'./exp/{args.experiment_curve}/configuration.json', 'r') as f:
            configuration = json.load(f)
        type = configuration['type']
        model = torch.load(f'./exp/{args.experiment_curve}/model.pth').cuda()
        inference(model, configuration['ws'], args.experiment_curve, type)


    # 병합
    merge(args.experiment_straight, args.experiment_curve)





def merge(experiment_straight, experiment_curve):
    make_dir('./answer')
    make_dir(f'./answer/{experiment_straight}_{experiment_curve}')
    answer_straight = pd.read_csv(f'./exp/{experiment_straight}/answer.csv')
    answer_curve = pd.read_csv(f'./exp/{experiment_curve}/answer.csv')
    answer = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
    answer.iloc[0:,1:1+20] = answer_straight.iloc[0:,1:1+20]
    answer.iloc[0:,21:21+20] = answer_curve.iloc[0:,21:21+20]
    answer.to_csv(f'./answer/{experiment_straight}_{experiment_curve}/answer.csv', index=False)






def inference(model, ws, experiment, type):
    # 이미 추론 결과가 있다면
    if os.path.exists(f'./exp/{experiment}/answer.csv'):
        reply = str(input('이미 추론 결과가 존재합니다. 다시 추론할지 선택하세요. (y/n): ')).lower().strip()
        if reply[0] == 'y':
            pass
        elif reply[0] == 'n':
            print('재추론하지 않습니다. 추론 코드를 종료합니다.')
            return
        else:
            warnings.warn('y 또는 n만 입력하세요.')
            inference(model, ws, experiment, type)
            return

    # 없다면, init variables
    data = np.load('./data/s_data.npy', allow_pickle=True) if type=='straight' else np.load('./data/c_data.npy', allow_pickle=True)
    answer = pd.read_csv('./data/answer_sample.csv')
    answer_idx = {30:1, 40:5, 50:9, 70:13, 100:17} if type=='straight' else {30:21, 40:25, 50:29, 70:33, 100:37}
    current_position = 0

    # test 해서 저장

    for start in tqdm(range(10000-ws+1,10000-ws+1+1999)):#tqdm(range(10001-ws)):#tqdm(range(10001-ws+1,10001-ws+1+1999)):
        model.eval()
        distance = torch.zeros((1,ws,1))
        lane = torch.zeros((1,ws,5))
        norm_target = torch.zeros((1,ws,5,4))

        # graph
        feature_wheel = torch.zeros((4, ws, 5, 8))
        feature_sensor = torch.zeros((2, ws , 5, 4))
        graph = dgl.heterograph({('wheel', 'front', 'wheel'): ([0,3],[3,0]),
                                 ('wheel', 'left', 'wheel'): ([0,1],[1,0]),
                                 ('wheel', 'right', 'wheel'): ([3,2],[2,3]),
                                 ('wheel', 'rear', 'wheel'): ([1,2],[2,1]),
                                 ('wheel', 'connect', 'sensor'):([0,1,2,3],[0,0,1,1]),
                                 ('sensor', 'connect', 'wheel'):([0,0,1,1],[0,1,2,3])})
        for i in range(ws):
            distance[0,i,...] = data[start+i]['distance']
            feature_wheel[:,i,...] = data[start+i]['graph'].nodes['wheel'].data['feature']
            feature_sensor[:,i,...] = data[start+i]['graph'].nodes['sensor'].data['feature']
            lane[0,i,...] = data[start+i]['lane'] if type=='straight' else torch.zeros(5)
            norm_target[0,i,...] = data[start+i]['norm_target']
        graph.nodes['wheel'].data['feature'] = feature_wheel.type(torch.float32)
        graph.nodes['sensor'].data['feature'] = feature_sensor.type(torch.float32)

        # to GPU
        graph = graph.to(torch.cuda.current_device())
        distance = distance.cuda()
        lane = lane.cuda()
        norm_target = norm_target.cuda()

        distances = [copy.deepcopy(distance), copy.deepcopy(distance), copy.deepcopy(distance), copy.deepcopy(distance), copy.deepcopy(distance)]
        lanes = [copy.deepcopy(lane), copy.deepcopy(lane), copy.deepcopy(lane), copy.deepcopy(lane), copy.deepcopy(lane)]
        graphs = [copy.deepcopy(graph), copy.deepcopy(graph), copy.deepcopy(graph), copy.deepcopy(graph), copy.deepcopy(graph)]
        norm_targets = [copy.deepcopy(norm_target), copy.deepcopy(norm_target), copy.deepcopy(norm_target), copy.deepcopy(norm_target), copy.deepcopy(norm_target)]
        for j, damper in enumerate([30,40,50,70,100]):
            model.eval()
            y_pred = model(distances[j], lanes[j], graphs[j], norm_targets[j], damper) # torch.tensor of (1, 4) shape
            answer.iloc[current_position,answer_idx[damper]:answer_idx[damper]+4] = y_pred[0,...].detach().cpu().tolist()
            for k in range(4):
                mean_std[j][k].update(y_pred[0,k].item())
                data[start+ws]['norm_target'][j][k] = (y_pred.detach().cpu().squeeze()[k].item()-mean_std[j][k].avg)/(mean_std[j][k].std+1e-3)
            # norm_y_pred = (y_pred.detach().cpu().squeeze()-mean_std_dic[type][damper][0])/mean_std_dic[type][damper][1]
            # if not (start+ws==12000):
                # data[start+ws]['norm_target'][j] = norm_y_pred.type(torch.float32)
        current_position += 1
    answer.to_csv(f'./exp/{experiment}/answer.csv', index=False)





if __name__ == '__main__':
    args = get_parser()
    main(args)

IndentationError: ignored

In [ ]:
# metric.py
import torch



class WeightedMAPE:
    def __init__(self):
        self.n = 0
        self.sum = 0

    def calc_mape(self, y_pred, y_true, position, n=1):
        self.n += n
        weight = 1.0001 + 0.0001*position
        self.sum += torch.sum(weight*torch.abs(y_true-y_pred)/torch.abs(y_true)).item()

    def get_mape(self):
        return self.sum * 1/self.n * 100



if __name__ == '__main__':
    w_mape = WeightedMAPE()

    y_true = torch.randn(32,4)
    y_pred = torch.randn(32,4)
    w_mape.calc_mape(y_pred, y_true, 399, y_pred.shape[0])

    y_true = torch.randn(32,4)
    y_pred = torch.randn(32,4)
    w_mape.calc_mape(y_pred, y_true, 400, y_pred.shape[0])

    mape = w_mape.get_mape()
    print(mape)

In [ ]:
# model.py
# library
import dgl
import dgl.nn.pytorch as dglnn
import torch.nn as nn
import torch
import torch.nn.functional as F
from einops import rearrange, repeat
import copy
from dgl.dataloading import GraphDataLoader


# local
from dataset import Dataset



# Railway models
class Model(nn.Module):
    def __init__(self,type='straight',aggregate='mean',rnn='transformer',ws=10,gpu=True,hidden_size=20,bidirectional=True,**kwargs):
        super().__init__()
        assert type in ['straight', 'curve']
        assert rnn in ['lstm', 'transformer']

        # variables
        self.type = type
        self.damper_dict = {30:0, 40:1, 50:2, 70:3, 100:4}
        self.ws = ws
        self.gpu = gpu
        self.rnn = rnn
        self.bi = 2 if bidirectional else 1

        # model
        self.feature_embedding_wheel = nn.Sequential(nn.Linear(40,20),
                                                     nn.LeakyReLU(),
                                                     nn.Linear(20,15))
        self.feature_embedding_sensor = nn.Sequential(nn.Linear(20,18),
                                                      nn.LeakyReLU(),
                                                      nn.Linear(18,15))
        self.conv1 = dglnn.HeteroGraphConv({'front':dglnn.GraphConv(15,8),
                                            'rear':dglnn.GraphConv(15,8),
                                            'right':dglnn.GraphConv(15,8),
                                            'left':dglnn.GraphConv(15,8),
                                            'connect':dglnn.GraphConv(15,8)},
                                           aggregate=aggregate)
        self.conv2 = dglnn.HeteroGraphConv({'front':dglnn.GraphConv(8,4),
                                            'rear':dglnn.GraphConv(8,4),
                                            'right':dglnn.GraphConv(8,4),
                                            'left':dglnn.GraphConv(8,4),
                                            'connect':dglnn.GraphConv(8,4)},
                                           aggregate=aggregate)
        self.reaky_relu = nn.LeakyReLU()
        self.damper_embedding = nn.Embedding(5,7)
        self.norm_target_embedding = nn.Sequential(nn.Linear(20,10),
                                                   nn.ReLU(),
                                                   nn.Linear(10,5))
        self.lane_embedding = nn.Linear(5,3)
        self.rnn_shape_embedding = nn.Linear(37,28) if type=='curve' else nn.Linear(40,28)
        if rnn == 'transformer':
            self.transformer = Transformer(dim=28,
                                           depth=3,
                                           heads=4,
                                           mlp_dim=14)
            self.linear = nn.Sequential(nn.Linear(ws*28, 128),
                                        nn.ReLU(),
                                        nn.Linear(128,32),
                                        nn.ReLU(),
                                        nn.Linear(32,4))
        elif rnn == 'lstm':
            self.lstm = nn.LSTM(input_size=28,
                                hidden_size=hidden_size,
                                num_layers=4,
                                batch_first=True,
                                bidirectional=bidirectional)
            self.linear = nn.Sequential(nn.Linear(ws*hidden_size*self.bi,int(ws*hidden_size*self.bi/2)),
                                        nn.ReLU(),
                                        nn.Linear(int(ws*hidden_size*self.bi/2),50),
                                        nn.ReLU(),
                                        nn.Linear(50,4))


    def forward(self, distance, lane, graph, norm_target, damper):
        '''
        distance : torch.tensor, (bs, ws, 1) shape
        graph : dgl.graph type
                graph contains ['wheel'] feature which has (4*bs,ws,5,8) shape and
                               ['sensor'] feature which has (2*bs,ws,5,4) shape.
        norm_target : torch.tensor, (bs, ws, 5, 4) shape
        damper : a scalar in [30,40,50,70,100]
        '''
        # graph
        graph.nodes['wheel'].data['feature'] = rearrange(graph.nodes['wheel'].data['feature'],
                                                         '(bs_and_wheels) ws dampers f -> (bs_and_wheels) ws (dampers f)') # (4*bs, ws, 40)
        graph.nodes['sensor'].data['feature'] = rearrange(graph.nodes['sensor'].data['feature'],
                                                         '(bs_and_sensors) ws dampers f -> (bs_and_sensors) ws (dampers f)') # (2*bs, ws, 40)
        graph.nodes['wheel'].data['feature'] = self.feature_embedding_wheel(graph.nodes['wheel'].data['feature']) # (4*bs, ws, 15)
        graph.nodes['sensor'].data['feature'] = self.feature_embedding_sensor(graph.nodes['sensor'].data['feature']) # (4*bs, ws, 15)
        h0 = {'wheel':graph.nodes['wheel'].data['feature'],
              'sensor':graph.nodes['sensor'].data['feature']}
        h1 = self.conv1(graph, h0)
        h1['wheel'] = self.reaky_relu(h1['wheel'])
        h1['sensor'] = self.reaky_relu(h1['sensor'])
        h2 = self.conv2(graph, h1)
        wheels = rearrange(h2['wheel'], '(bs wheels) ws d -> bs ws (wheels d)', bs=distance.shape[0]) # (bs, ws, 16)
        sensors = rearrange(h2['sensor'], '(bs sensors) ws d -> bs ws (sensors d)', bs=distance.shape[0]) # (bs, ws, 8)

        # other variables
        damper = self.damper_embedding(torch.LongTensor([self.damper_dict[damper]]).cuda()) # (1, 7)
        damper = torch.squeeze(damper) # (7)
        damper = repeat(damper, 'd -> bs ws d', bs=distance.shape[0], ws=self.ws) # (bs, ws, 7)
        norm_target = rearrange(norm_target, 'bs ws dampers targets -> bs ws (dampers targets)') # (bs, ws, 20)
        norm_target = self.norm_target_embedding(norm_target) # (bs, ws, 5)

        # concat all variables
        if self.type == 'straight':
            lane = self.lane_embedding(lane) # (bs, ws, 3)
            x = torch.cat([distance, lane, wheels, sensors, norm_target, damper], dim=-1) # (bs, ws, 1+3+16+8+5+7) = (bs, ws, 40)
        else:
            x = torch.cat([distance, wheels, sensors, norm_target, damper], dim=-1) # (bs, ws, 1+16+8+5+7) = (bs, ws, 37)

        # forward to RNN model and inference
        x = self.rnn_shape_embedding(x) # (bs, ws, 28)
        if self.rnn == 'lstm':
            x,_ = self.lstm(x)
            x = rearrange(x, 'bs ws (hs bi) -> bs (ws hs bi)', bi=self.bi) # (bs, ws*hidden_size*bi)
            x = self.linear(x)
        elif self.rnn == 'transformer':
            x = self.transformer(x)
            x = rearrange(x, 'bs ws d -> bs (ws d)') # (bs, 280)
            x = self.linear(x)

        return x # (bs, 4)





# Transformers
class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.scale = dim ** -0.5

        self.to_qkv = nn.Linear(dim, dim * 3, bias=False)
        self.to_out = nn.Linear(dim, dim)

    def forward(self, x, mask = None):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x)
        q, k, v = rearrange(qkv, 'b n (qkv h d) -> qkv b h n d', qkv=3, h=h)

        dots = torch.einsum('bhid,bhjd->bhij', q, k) * self.scale

        if mask is not None:
            mask = F.pad(mask.flatten(1), (1, 0), value = True)
            assert mask.shape[-1] == dots.shape[-1], 'mask has incorrect dimensions'
            mask = mask[:, None, :] * mask[:, :, None]
            dots.masked_fill_(~mask, float('-inf'))
            del mask

        attn = dots.softmax(dim=-1)

        out = torch.einsum('bhij,bhjd->bhid', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out =  self.to_out(out)
        return out

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, mlp_dim):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Residual(PreNorm(dim, Attention(dim, heads = heads))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim)))
            ]))

    def forward(self, x, mask=None):
        for attn, ff in self.layers:
            x = attn(x, mask=mask)
            x = ff(x)
        return x




if __name__ == '__main__':
    # Curve Model Test
    ## data
    dataset = Dataset('curve', 10, 'train')
    dataloader = GraphDataLoader(dataset, batch_size=32, shuffle=True, drop_last=False)
    distance, lane, graph, norm_target, target = next(iter(dataloader))
    distance = distance.cuda()
    lane = lane.cuda()
    graph = graph.to(torch.cuda.current_device())
    norm_target = norm_target.cuda()
    target = target.cuda()

    ## model
    curve_trans = Model(type='curve').cuda()
    curve_lstm = Model(type='curve', rnn='lstm')
    pred_curve_trans = curve_trans(distance, lane, copy.deepcopy(graph), norm_target, 30)
    pred_curve_lstm = curve_trans(distance, lane, copy.deepcopy(graph), norm_target, 40)
    print(f'Curve Transformer output : {pred_curve_trans.shape}')
    print(f'Curve LSTM output : {pred_curve_lstm.shape}')
    del curve_lstm, curve_trans, dataset, dataloader


    # Straight Model Test
    ## data
    dataset = Dataset('straight', 10, 'train')
    dataloader = GraphDataLoader(dataset, batch_size=32, shuffle=True, drop_last=False)
    distance, lane, graph, norm_target, target = next(iter(dataloader))
    distance = distance.cuda()
    lane = lane.cuda()
    graph = graph.to(torch.cuda.current_device())
    norm_target = norm_target.cuda()
    target = target.cuda()

    ## model
    straight_trans = Model(type='straight').cuda()
    straight_lstm = Model(type='straight', rnn='lstm')
    pred_straight_trans = straight_trans(distance, lane, copy.deepcopy(graph), norm_target, 50)
    pred_straight_lstm = straight_trans(distance, lane, copy.deepcopy(graph), norm_target, 100)
    print(f'straight Transformer output : {pred_straight_trans.shape}')
    print(f'Straight LSTM output : {pred_straight_lstm.shape}')

In [ ]:
'''
이 코드에서는 dgl.data 또는 torch.utils.data의 DataLoader, Dataset을 사용하지 않습니다.
미리 모든 데이터셋을 프로세싱하여 로컬에 저장합니다.
'''
# preprocess.py
# library
import pandas as pd
import numpy as np
import torch
import dgl
from tqdm import tqdm


# local
from utils import normalize_KRRI_data



def processing(type=None):
    assert type in ['straight', 'curve'], 'Data type error'
    type = type[0]

    # data : [lenth x 39 cols]
    data_30 = normalize_KRRI_data(pd.read_csv(f'./data/data_{type}30.csv'))
    data_40 = normalize_KRRI_data(pd.read_csv(f'./data/data_{type}40.csv'))
    data_50 = normalize_KRRI_data(pd.read_csv(f'./data/data_{type}50.csv'))
    data_70 = normalize_KRRI_data(pd.read_csv(f'./data/data_{type}70.csv'))
    data_100 = normalize_KRRI_data(pd.read_csv(f'./data/data_{type}100.csv'))

    # lane data
    data_lane = pd.read_csv(f'./data/lane_data_{type}.csv')
    data_lane = (data_lane-data_lane.mean())/data_lane.std()

    # process
    data_list = []
    for idx in tqdm(range(len(data_30)),desc=f'{type} 데이터에서 전처리중...'):
        # graph
        '''
        ~~~~~~~~~~~~~
        (n) : wheel
        [n] : sensor
        ~~~~~~~~~~~~~
        ┌(0) ㅡ (3)┐: front
        | |      | |
        |[0] ㅡ [1]ㅣ
        | |      | |
        └(1) ㅡ (2)┘ : rear
        '''
        graph = dgl.heterograph({('wheel', 'front', 'wheel'): ([0,3],[3,0]),
                                ('wheel', 'left', 'wheel'): ([0,1],[1,0]),
                                ('wheel', 'right', 'wheel'): ([3,2],[2,3]),
                                ('wheel', 'rear', 'wheel'): ([1,2],[2,1]),
                                ('wheel', 'connect', 'sensor'):([0,1,2,3],[0,0,1,1]),
                                ('sensor', 'connect', 'wheel'):([0,0,1,1],[0,1,2,3])})

        ## wheel feature
        feature_wheel_0 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,3,9,15,19,23,25]])[None,...],
                                     torch.tensor(data_40.iloc[idx,[1,2,3,9,15,19,23,25]])[None,...],
                                     torch.tensor(data_50.iloc[idx,[1,2,3,9,15,19,23,25]])[None,...],
                                     torch.tensor(data_70.iloc[idx,[1,2,3,9,15,19,23,25]])[None,...],
                                     torch.tensor(data_100.iloc[idx,[1,2,3,9,15,19,23,25]])[None,...]),
                                    dim=0) # 좌전륜 # (5,8) shape
        feature_wheel_1 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,5,11,16,20,27,29]])[None,...],
                                     torch.tensor(data_40.iloc[idx,[1,2,5,11,16,20,27,29]])[None,...],
                                     torch.tensor(data_50.iloc[idx,[1,2,5,11,16,20,27,29]])[None,...],
                                     torch.tensor(data_70.iloc[idx,[1,2,5,11,16,20,27,29]])[None,...],
                                     torch.tensor(data_100.iloc[idx,[1,2,5,11,16,20,27,29]])[None,...]),
                                    dim=0) # 좌후륜 # (5,8) shape
        feature_wheel_2 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,8,14,18,22,28,30]])[None,...],
                                     torch.tensor(data_40.iloc[idx,[1,2,8,14,18,22,28,30]])[None,...],
                                     torch.tensor(data_50.iloc[idx,[1,2,8,14,18,22,28,30]])[None,...],
                                     torch.tensor(data_70.iloc[idx,[1,2,8,14,18,22,28,30]])[None,...],
                                     torch.tensor(data_100.iloc[idx,[1,2,8,14,18,22,28,30]])[None,...]),
                                    dim=0) # 우후륜 # (5,8) shape
        feature_wheel_3 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,6,12,17,21,24,26]])[None,...],
                                     torch.tensor(data_40.iloc[idx,[1,2,6,12,17,21,24,26]])[None,...],
                                     torch.tensor(data_50.iloc[idx,[1,2,6,12,17,21,24,26]])[None,...],
                                     torch.tensor(data_70.iloc[idx,[1,2,6,12,17,21,24,26]])[None,...],
                                     torch.tensor(data_100.iloc[idx,[1,2,6,12,17,21,24,26]])[None,...]),
                                    dim=0) # 우전륜 # (5,8) shape
        feature_wheel = torch.cat((feature_wheel_0[None,...],
                                   feature_wheel_1[None,...],
                                   feature_wheel_2[None,...],
                                   feature_wheel_3[None,...]),
                                  dim=0).type(torch.float32) # (4,5,8) shape

        ## sensor feature
        feature_sensor_0 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,4,10]])[None,...],
                                      torch.tensor(data_40.iloc[idx,[1,2,4,10]])[None,...],
                                      torch.tensor(data_50.iloc[idx,[1,2,4,10]])[None,...],
                                      torch.tensor(data_70.iloc[idx,[1,2,4,10]])[None,...],
                                      torch.tensor(data_100.iloc[idx,[1,2,4,10]])[None,...]),
                                      dim=0) # 좌sensor # (5,4) shape
        feature_sensor_1 = torch.cat((torch.tensor(data_30.iloc[idx,[1,2,7,13]])[None,...],
                                      torch.tensor(data_40.iloc[idx,[1,2,7,13]])[None,...],
                                      torch.tensor(data_50.iloc[idx,[1,2,7,13]])[None,...],
                                      torch.tensor(data_70.iloc[idx,[1,2,7,13]])[None,...],
                                      torch.tensor(data_100.iloc[idx,[1,2,7,13]])[None,...]),
                                      dim=0) # 우sensor # (5,4) shape
        feature_sensor = torch.cat((feature_sensor_0[None,...],
                                    feature_sensor_1[None,...]),
                                    dim=0).type(torch.float32) # (2,5,4) shape
        ## feature 삽입
        graph.nodes['wheel'].data['feature'] = feature_wheel
        graph.nodes['sensor'].data['feature'] = feature_sensor

        # 기타 variables
        ## distance
        distance = data_30.iloc[idx,0] # scalar
        ## norm_target
        norm_target = torch.tensor([data_30.iloc[idx,-8:-4].values,
                                    data_40.iloc[idx,-8:-4].values,
                                    data_50.iloc[idx,-8:-4].values,
                                    data_70.iloc[idx,-8:-4].values,
                                    data_100.iloc[idx,-8:-4].values]).type(torch.float32) # (5, 4) shape
        ## target
        target = torch.tensor([data_30.iloc[idx,-4:].values,
                               data_40.iloc[idx,-4:].values,
                               data_50.iloc[idx,-4:].values,
                               data_70.iloc[idx,-4:].values,
                               data_100.iloc[idx,-4:].values]).type(torch.float32) # (5, 4) shape

        ## lane
        lane = torch.tensor(data_lane.iloc[idx,1:].values).type(torch.float32) if type=='s' else [] # 5 length for straight

        # append
        data_list.append({'distance':distance,
                          'graph':graph,
                          'lane':lane,
                          'norm_target':norm_target,
                          'target':target})

    # save
    np.save(f'./data/{type}_data.npy', np.array(data_list))





def get_mean_std(df):
    mean = torch.zeros(4)
    std = torch.zeros(4)
    mean[0] = df['YL_M1_B1_W1'][0:10001].mean()
    mean[1] = df['YR_M1_B1_W1'][0:10001].mean()
    mean[2] = df['YL_M1_B1_W2'][0:10001].mean()
    mean[3] = df['YR_M1_B1_W2'][0:10001].mean()
    std[0] = df['YL_M1_B1_W1'][0:10001].std()
    std[1] = df['YR_M1_B1_W1'][0:10001].std()
    std[2] = df['YL_M1_B1_W2'][0:10001].std()
    std[3] = df['YR_M1_B1_W2'][0:10001].std()
    mean = mean.type(torch.float32)
    std = std.type(torch.float32)
    return mean, std




if __name__ == '__main__':
    processing('curve')
    processing('straight')

In [ ]:
# utils.py
# library
import pandas as pd
import os
from matplotlib import pyplot as plt
from collections.abc import Iterable
import numpy as np
import copy


def normalize_KRRI_data(df):
    '''
    - df의 0번 열은 timestamp
    - 1번부터 30번 열은 features
    - 31,32,33,34번 열은 target이다.
    - target 열은 추론해야하는 부분이 모두 0이므로 평균 분산 계산에서는 따지지 말아야한다
    '''
    feature = df.iloc[0:,0:31] # 시간도 그냥 표준화함
    feature = (feature-feature.mean())/feature.std()
    norm_target = copy.deepcopy(df.iloc[0:,31:])
    norm_target.rename(columns={'YR_M1_B1_W2':'norm_YR_M1_B1_W2',
                                'YL_M1_B1_W2':'norm_YL_M1_B1_W2',
                                'YR_M1_B1_W1':'norm_YR_M1_B1_W1',
                                'YL_M1_B1_W1':'norm_YL_M1_B1_W1'},
                       inplace=True)
    norm_target.iloc[0:10001,0:] = (norm_target.iloc[0:10001,0:]-norm_target.iloc[0:10001,0:].mean())/norm_target.iloc[0:10001,0:].std()
    target = df.iloc[0:,31:]
    processed_df = pd.concat([feature, norm_target, target], axis=1, ignore_index=False)
    return processed_df



def make_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)



class AverageMeter(object):
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.sum_2 = 0  # sum of squares
        self.count = 0
        self.std = 0

    def update(self, val, n=1):
        if val != None:  # update if val is not None
            self.val = val
            self.sum += val * n
            self.sum_2 += val ** 2 * n
            self.count += n
            self.avg = self.sum / self.count
            self.std = np.sqrt(self.sum_2 / self.count - self.avg ** 2)
        else:
            pass




class Logger(object):
    def __init__(self, path, int_form=':03d', float_form=':.4f'):
        self.path = path
        self.int_form = int_form
        self.float_form = float_form
        self.width = 0

    def __len__(self):
        try:
            return len(self.read())
        except:
            return 0

    def write(self, values):
        if not isinstance(values, Iterable):
            values = [values]
        if self.width == 0:
            self.width = len(values)
        assert self.width == len(values), 'Inconsistent number of items.'
        line = ''
        for v in values:
            if isinstance(v, int):
                line += '{{{}}} '.format(self.int_form).format(v)
            elif isinstance(v, float):
                line += '{{{}}} '.format(self.float_form).format(v)
            elif isinstance(v, str):
                line += '{} '.format(v)
            else:
                raise Exception('Not supported type.', v)
        with open(self.path, 'a') as f:
            f.write(line[:-1] + '\n')

    def read(self):
        with open(self.path, 'r') as f:
            log = []
            for line in f:
                values = []
                for v in line.split(' '):
                    try:
                        v = float(v)
                    except:
                        pass
                    values.append(v)
                log.append(values)
        return log



def draw_curve(work_dir, train_logger, test_logger):
        train_logger = train_logger.read()
        test_logger = test_logger.read()
        epoch, train_loss = zip(*train_logger)
        epoch,test_loss = zip(*test_logger)

        plt.plot(epoch, train_loss, color='blue', label="Train Loss")
        plt.plot(epoch, test_loss, color='red', label="Test Loss")
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title("Loss Curve")
        plt.legend()
        plt.grid(True)
        plt.savefig(work_dir + '/loss_curve.png')
        plt.close()




if __name__ == '__main__':
    df = pd.read_csv('./data/data_c30.csv')
    df = normalize_KRRI_data(df)
    print(df.head(10))